In [ ]:
import math
from functools import partial
import torch
import torch.nn.utils.parametrize as parametrize
from torch import nn

In [ ]:
class LoRAParametrization(nn.Module):
  def __init__(self, fan_in, fan_out, fan_in_fan_out=False, rank=4, lora_dropout_p=0.0, lora_alpha=1):
    super().__init__()
    self.swap=(lambda x: (x[1],x[0])) if fan_in_fan_out else (lambda x: x)
    self.lora_A = nn.Parameter(torch.zeros(self.swap((rank,fan_in))))
    self.lora_B = nn.Parameter(torch.zeros(self.swap((fan_out, rank))))
    nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
    self.lora_alpha, self.rank = lora_alpha, rank
    self.scaling = lora_alpha / rank
    self.lora_dropout = nn.Dropout(p=lora_dropout_p) if lora_dropout_p > 0 else lambda x: x
    self.dropout_fn = self._dropout if lora_dropout_p > 0 else lambda x: x
    self.register_buffer("lora_dropout_mask", torch.ones(self.swap((1, fan_in)), dtype=self.lora_A.dtype))
    self.forward_fn = self.lora_forward

  def _dropout(self,A):
      return A * self.lora_dropout(self.lora_dropout_mask)
  def lora_forward(self, X):
      return X + torch.matmul(*self.swap((self.lora_B, self.dropout_fn(self.lora_A)))).view(X.shape) * self.scaling

  def forward(self, X):
      return self.forward_fn(X)

  def disable_lora(self):
      self.forward_fn = lambda x: x

  def enable_lora(self):
      self.forward_fn = self.lora_forward

  @classmethod
  def from_linear(cls, layer, rank=4, lora_dropout_p=0.0, lora_alpha=1):
        fan_out, fan_in = layer.weight.shape
        return cls(
            fan_in, fan_out, fan_in_fan_out=False, rank=rank, lora_dropout_p=lora_dropout_p, lora_alpha=lora_alpha
        )

  @classmethod
  def from_conv2d(cls, layer, rank=4, lora_dropout_p=0.0, lora_alpha=1):
        fan_out, fan_in = layer.weight.view(layer.weight.shape[0], -1).shape
        return cls(
            fan_in, fan_out, fan_in_fan_out=False, rank=rank, lora_dropout_p=lora_dropout_p, lora_alpha=lora_alpha
        )

  @classmethod
  def from_embedding(cls, layer, rank=4, lora_dropout_p=0.0, lora_alpha=1):
        fan_in, fan_out = layer.weight.shape
        return cls(
            fan_in, fan_out, fan_in_fan_out=True, rank=rank, lora_dropout_p=lora_dropout_p, lora_alpha=lora_alpha
        )



In [ ]:
default_lora_config = {
    nn.Linear: {
        "weight": partial(LoRAParametrization.from_linear, rank=4),
    },
}


def apply_lora(layer, register=True, merge=False, lora_config=default_lora_config):
    if register:
        if type(layer) in lora_config:
            for attr_name, parametrization in lora_config[type(layer)].items():
                parametrize.register_parametrization(layer, attr_name, parametrization(layer))
    else:
        if hasattr(layer, "parametrizations"):
            for attr_name in list(layer.parametrizations.keys()):
                parametrize.remove_parametrizations(layer, attr_name, leave_parametrized=merge)


def add_lora(model, lora_config=default_lora_config):
    model.apply(partial(apply_lora, lora_config=lora_config))


def add_lora_by_name(model, target_module_names, lora_config=default_lora_config):
    for name, layer in model.named_modules():
        if any([m in name for m in target_module_names]):
            add_lora(layer, lora_config=lora_config)


def merge_lora(model):
    model.apply(partial(apply_lora, register=False, merge=True))
#parametrize.remove_parametrizations(layer, attr_name, leave_parametrized=True)

def remove_lora(model):
    model.apply(partial(apply_lora, register=False, merge=False))

In [ ]:
def apply_to_lora(fn):

  def apply_fn(layer):
    if isinstance(layer, LoRAParametrization):
      fn(layer)
  return apply_fn

enable_lora = lambda model: model.apply(apply_to_lora(lambda x: x.enable_lora()))
disable_lora = lambda model: model.apply(apply_to_lora(lambda x: x.disable_lora()))

In [ ]:
def name_is_lora(name):
    return (
        len(name.split(".")) >= 4
        and (name.split(".")[-4]) == "parametrizations"
        and name.split(".")[-1] in ["lora_A", "lora_B"]
    )

    #return True or False


def name_is_bias(name):
    return name.split(".")[-1] == "bias"


def get_params_by_name(model, print_shapes=False, name_filter=None):
    for n, p in model.named_parameters():
        if name_filter is None or name_filter(n):
            if print_shapes:
                print(n, p.shape)
            yield p


def get_lora_params(model, print_shapes=False):
    return get_params_by_name(model, print_shapes=print_shapes, name_filter=name_is_lora)


def get_bias_params(model, print_shapes=False):
    return get_params_by_name(model, print_shapes=print_shapes, name_filter=name_is_bias)


def get_lora_state_dict(model):
    return {k: v for k, v in model.state_dict().items() if name_is_lora(k)}


def _prepare_for_multiple_lora(lora_layer):
    lora_layer.lora_As = []
    lora_layer.lora_Bs = []


def _append_lora(lora_layer):
    lora_layer.lora_As.append(nn.Parameter(lora_layer.lora_A.clone()))
    lora_layer.lora_Bs.append(nn.Parameter(lora_layer.lora_B.clone()))


def load_multiple_lora(model, lora_state_dicts):
    model.apply(apply_to_lora(_prepare_for_multiple_lora))
    for state_dict in lora_state_dicts:
        _ = model.load_state_dict(state_dict, strict=False)
        model.apply(apply_to_lora(_append_lora))
    return model


def _select_lora(lora_layer, index):
    lora_layer.lora_A = lora_layer.lora_As[index]
    lora_layer.lora_B = lora_layer.lora_Bs[index]


def select_lora(model, index):
    model.apply(apply_to_lora(lambda x: _select_lora(x, index)))
    return model


def tie_weights(linear: nn.Linear, embedding: nn.Embedding):
    embedding.parametrizations.weight.original = linear.parametrizations.weight.original
    embedding.parametrizations.weight[0].lora_A = linear.parametrizations.weight[0].lora_B
    embedding.parametrizations.weight[0].lora_B = linear.parametrizations.weight[0].lora_A


def untie_weights(linear: nn.Linear, embedding: nn.Embedding):
    embedding.parametrizations.weight.original = nn.Parameter(embedding.weight.original.clone())
    embedding.parametrizations.weight[0].lora_A = nn.Parameter(embedding.parametrizations.weight[0].lora_A.clone())
    embedding.parametrizations.weight[0].lora_B = nn.Parameter(embedding.parametrizations.weight[0].lora_B.clone())

In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(in_features=5, out_features=7),
    torch.nn.Linear(in_features=7, out_features=3),
)

x = torch.randn(1, 5)
y = model(x)
print(y)
Y0 = y #store the output tensor

tensor([[-0.8080,  0.2055, -0.3166]], grad_fn=<AddmmBackward0>)


In [ ]:
add_lora(model)
y = model(x)
assert torch.allclose(y, Y0)
#check if they are the same

In [ ]:
model.apply(apply_to_lora(lambda x: torch.nn.init.ones_(x.lora_B)))
y = model(x)
print(y)
assert not torch.allclose(y, Y0)
Y1 = y

tensor([[-0.9465, -0.2872, -0.7695]], grad_fn=<AddmmBackward0>)


In [ ]:
disable_lora(model)
y = model(x)
assert torch.allclose(y, Y0)

In [ ]:
enable_lora(model)
y = model(x)
assert torch.allclose(y, Y1)

In [ ]:
state_dict_to_save = get_lora_state_dict(model)
state_dict_to_save.keys()

dict_keys(['0.parametrizations.weight.0.lora_A', '0.parametrizations.weight.0.lora_B', '1.parametrizations.weight.0.lora_A', '1.parametrizations.weight.0.lora_B'])

In [ ]:
remove_lora(model)

In [ ]:
add_lora(model)

In [ ]:
_ = model.load_state_dict(state_dict_to_save, strict=False)
y = model(x)
assert torch.allclose(y, Y1)

In [ ]:
merge_lora(model)
y = model(x)
assert torch.allclose(y, Y1)

In [ ]:
model

Sequential(
  (0): Linear(in_features=5, out_features=7, bias=True)
  (1): Linear(in_features=7, out_features=3, bias=True)
)

In [ ]:
#training
model = torch.nn.Linear(in_features=5, out_features=3)
add_lora(model)

In [ ]:
parameters = [
    {"params": list(get_lora_params(model))},
]
optimizer = torch.optim.AdamW(parameters, lr=1e-3)

In [ ]:
model.apply(apply_to_lora(lambda x: torch.nn.init.normal_(x.lora_A)))
model.apply(apply_to_lora(lambda x: torch.nn.init.normal_(x.lora_B)))

state_dict = model.state_dict()
lora_state_dict = {k: v for k, v in state_dict.items() if name_is_lora(k)}

In [ ]:
add_lora(model)
_ = model.load_state_dict(lora_state_dict, strict=False)
merge_lora(model)

In [ ]:
remove_lora(model)

In [ ]:
add_lora(model)

In [ ]:
lora_state_dict_0 = lora_state_dict
lora_state_dict_1 = {k: torch.ones_like(v) for k, v in lora_state_dict.items()}
lora_state_dict_2 = {k: torch.zeros_like(v) for k, v in lora_state_dict.items()}
lora_state_dicts = [lora_state_dict_0, lora_state_dict_1, lora_state_dict_2]

load_multiple_lora(model, lora_state_dicts)

Y0 = select_lora(model, 0)(x)
Y1 = select_lora(model, 1)(x)
Y2 = select_lora(model, 2)(x)

In [ ]:
Y0, Y1, Y2

(tensor([[-0.1950,  0.3060,  1.8896]], grad_fn=<AddmmBackward0>),
 tensor([[-0.2529, -0.4868,  0.9578]], grad_fn=<AddmmBackward0>),
 tensor([[0.2500, 0.0162, 1.4607]], grad_fn=<AddmmBackward0>))

In [ ]:
remove_lora(model)
init_state_dict = model.state_dict()
for state_dict in lora_state_dicts:
    remove_lora(model)
    _ = model.load_state_dict(init_state_dict, strict=False)
    add_lora(model)
    _ = model.load_state_dict(state_dict, strict=False)
    merge_lora(model)
    y = model(x)
    print(y)

tensor([[-0.1950,  0.3060,  1.8896]], grad_fn=<AddmmBackward0>)
tensor([[-0.2529, -0.4868,  0.9578]], grad_fn=<AddmmBackward0>)
tensor([[0.2500, 0.0162, 1.4607]], grad_fn=<AddmmBackward0>)
